In [ ]:
# Imports
import cProfile
import pstats
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import WCSAxes
from astropy.coordinates import SkyCoord, FK5
from spectral_cube import SpectralCube
from velocity_tools import extract_streamline, gradient_descent, stream_lines_grad, outputs, errors
from velocity_tools import stream_lines # won't use this directly, but needed to compare with stream_lines_grad
import os
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
# allow jax print statements to show up in the notebook
import jax.numpy as jnp
from jax import value_and_grad
import pandas as pd
import optax

import warnings
warnings.filterwarnings('ignore', message='.*PV2_1.*')
warnings.filterwarnings('ignore', message='.*PV2_2.*')
warnings.filterwarnings('ignore', message='.*TIMESYS.*')

# Settings
hltau_c= SkyCoord("4h31m38.43s", "+18d13m57.19s", frame='fk5')
hltau_ref = hltau_c.skyoffset_frame()
iras2a_c = SkyCoord("3h28m55.569s", "+31d14m37.025s", frame='fk5')
iras2a_ref = iras2a_c.skyoffset_frame()
distance_hltau = 147 #parsecs
distance_iras2a = 293 #parsecs
v_lsr = 7.5 #km/s
# choose which distance
distance = distance_iras2a

# cubefile = 'test_data/HLTau/HLTAU_HCOp32.fits'a
# file_Tpeak = 'test_data/HLTau/HLTAU_HCOp32_Tpeak.fits'
cubefile = 'test_data/IRAS2A/D2CO_streamer_cluster_data.fits'
file_Tpeak = 'test_data/IRAS2A/D2CO_streamer_cluster_tpeak.fits'

# some constants
G = 6.67430e-11 * (1e-3)**2 * (1.988416e30) / (1.4959787e11) # in au (km/s)^2 * Msol^-1
au_in_km = 1.4959787e8 #km


## 1. Prepare 1D streamer emission from cube

The cube should contain only streamer emission. See 'how do I isolate my streamer emission?'.

In [ ]:
# get the spectralcube object from the data using spectral-cube
hdu = fits.open(cubefile)[0]
cube = SpectralCube.read(hdu).with_spectral_unit(u.km/u.s, rest_value=hdu.header['RESTFRQ']*u.Hz)

# TODO: this streamer extraction should be replaced with clustering-based streamer extraction

# extract the subcube with the streamer (we got this from the tipsy tutorial, no need to plot and in fact no need to do this)
## Limits for extracting subcube with streamer


vmin = 6
vmax = 8
xmin = -5
xmax = 5
ymin = -12
ymax = 0.5
rms_thresh = 4

# extract the streamer subcube
info_header = cube.header  # header of the cube, contains required information
x_conv_fac = 1/60/60/info_header['CDELT1']
xmin_p = int((xmin*x_conv_fac)+info_header['CRPIX1'])
xmax_p = int((xmax*x_conv_fac)+info_header['CRPIX1'])
y_conv_fac = 1/60/60/info_header['CDELT2']
ymin_p = int((ymin*y_conv_fac)+info_header['CRPIX2'])
ymax_p = int((ymax*y_conv_fac)+info_header['CRPIX2'])
vunit = cube.spectral_axis.unit
## Note: a check can be added to see if requested limits are within the limits of the cube itself

streamer_cubev = cube.spectral_slab(vmin*vunit,vmax*vunit)    # Selecting velocities
streamer_cubevc = streamer_cubev[:,min(ymin_p,ymax_p):max(ymin_p,ymax_p)   
                    ,min(xmin_p,xmax_p):max(xmin_p,xmax_p)]   # Selecting pixels
#     print(min(ymin_p,ymax_p),max(ymin_p,ymax_p),min(xmin_p,xmax_p),max(xmin_p,xmax_p)) 
streamer_cube = streamer_cubevc.with_mask(streamer_cubevc > rms_thresh*streamer_cubevc.mad_std())  # Removing low flux values 

print('vunit:', vunit)
print('Streamer cube spectral axis limits:', streamer_cube.spectral_axis.min(), streamer_cube.spectral_axis.max())

n_points = 10 # the number of points we want to reduce the data to


# Extract 1D streamline from the data cube
pc_coords, pc_means, pc_stds = extract_streamline.reduce_to_1D(streamer_cube, iras2a_c, n_elements=n_points)
print(f"point cloud velocities (km/s): {pc_coords[2]}")

# Prepare data for gradient descent
ra_data = pc_means[0] # offsets in arcsec
dec_data = pc_means[1] # offsets in arcsec
v_data = pc_means[2]   # velocities in km/s (rel to vlsr)
print(f"data velocities (km/s): {v_data}")

ra_sigma = pc_stds[0]
dec_sigma = pc_stds[1]
v_sigma = pc_stds[2]

data = (ra_data, dec_data, v_data)
uncertainties = (ra_sigma, dec_sigma, v_sigma)

# plot the extracted morphology
outputs.plot_morphology(
    ra_data=ra_data,
    dec_data=dec_data,
    ra_sigma=ra_sigma,
    dec_sigma=dec_sigma,
    pc_coords=pc_coords,
    show=True,
)


## 2. Initial Guess

First set your input params in the format required (separate params to optimise and params to stay fixed). Use units.

In [ ]:
# Parameters to optimize
initial_opt_params = {
    'r0': 1500.0 * u.au,  # au
    'theta0': 40.0 * u.deg,  # degrees
    'phi0': 100.0 * u.deg,  # degrees
    'log_omega': np.log(5e-13),  # log(1/s)
    'v_r0': 0.1 * u.km / u.s # km/s
}

# IRAS2A
fixed_params = {
    'mass': 4.0 * u.Msun,  # solar masses
    'inc': -45.0 * u.deg,  # degrees
    'pa': 194.0 * u.deg,  # degrees
    'rmin': 50.0 * u.au,  # au
    'deltar': 40.0 * u.au,  # au
    'v_lsr': 7.5 * u.km / u.s,  # km/s (systemic velocity)
}

Now we run the forward model for the initial guess, and calculate the loss (i.e. the 'difference' between the model and data streamline)

In [ ]:
# forward model (same inputs as chi2_loss)
model_params, initial_opt_params, fixed_params = gradient_descent.prepare_model_params(initial_opt_params, fixed_params)
ra_model, dec_model, v_model, valid_mask_model, err = gradient_descent.forward_model(
    model_params, distance
)
err.throw()

# match model to data exactly as chi2_loss does
ra_model_interp, dec_model_interp, v_model_interp, valid, model_keep, _dmetric_model, _ = gradient_descent.checked_match_model_to_data_curve(
    ra_model, dec_model, v_model, valid_mask_model, ra_data, dec_data
)

print(initial_opt_params)
print(fixed_params)
print(f"ra_model_interp = {ra_model_interp}")
print(f"ra_model_interp (valid) = {ra_model_interp[valid]}")
print(f"retained {int(jnp.sum(valid))}/{len(valid)} data points after overlap filtering")

# ---- Manual chi2_loss calculation (same logic as gradient_descent.chi2_loss) ----
# Coerce to float64 and floor sigmas to avoid division by zero
ra_data_f = jnp.asarray(ra_data, dtype=jnp.float64)
dec_data_f = jnp.asarray(dec_data, dtype=jnp.float64)
v_data_f = jnp.asarray(v_data, dtype=jnp.float64)

ra_sigma_f = jnp.asarray(ra_sigma, dtype=jnp.float64)
dec_sigma_f = jnp.asarray(dec_sigma, dtype=jnp.float64)
v_sigma_f = jnp.asarray(v_sigma, dtype=jnp.float64)

eps = jnp.asarray(1e-8, dtype=jnp.float64)
ra_sigma_safe = jnp.maximum(ra_sigma_f, eps)
dec_sigma_safe = jnp.maximum(dec_sigma_f, eps)
v_sigma_safe = jnp.maximum(v_sigma_f, eps)



r_data, theta_data = extract_streamline.cartesian_to_polar(ra_data_f, dec_data_f)
r_model, theta_model = extract_streamline.cartesian_to_polar(ra_model_interp, dec_model_interp)

dtheta = extract_streamline.wrap_to_pi(theta_data - theta_model)

sigma_r = jnp.sqrt(ra_sigma**2 + dec_sigma**2)
r_eps = 1e-8
r_safe = jnp.maximum(jnp.abs(r_data), r_eps)
sigma_theta = jnp.sqrt(((dec_data * dec_sigma)**2 + (ra_data * ra_sigma)**2)) / (r_safe**2)
sigma_theta = jnp.maximum(sigma_theta, r_eps)

# Only compute chi2 on valid/retained data points
chi2_r = jnp.sum((((r_data[valid] - r_model[valid]) / sigma_r[valid]) ** 2))
chi2_theta = jnp.sum(((dtheta[valid] / sigma_theta[valid]) ** 2))
chi2_v = jnp.sum((((v_data_f[valid] - v_model_interp[valid]) / v_sigma_safe[valid]) ** 2))
chi2_total = chi2_r + chi2_theta + chi2_v # + chi2_penalty


print(
    f"Chi2 r: {chi2_r:.2f}, Chi2 theta: {chi2_theta:.2f}, "
    f"Chi2 v: {chi2_v:.2f}, Total: {chi2_total:.2f}"
)

# do the plot using the morphology plotting function
outputs.plot_morphology(
    ra_model=jnp.asarray(ra_model, dtype=jnp.float64),
    dec_model=jnp.asarray(dec_model, dtype=jnp.float64),
    ra_data=jnp.asarray(ra_data, dtype=jnp.float64),
    dec_data=jnp.asarray(dec_data, dtype=jnp.float64),
    ra_sigma=jnp.asarray(ra_sigma, dtype=jnp.float64),
    dec_sigma=jnp.asarray(dec_sigma, dtype=jnp.float64),
    ra_model_interp=jnp.asarray(ra_model_interp[valid], dtype=jnp.float64),
    dec_model_interp=jnp.asarray(dec_model_interp[valid], dtype=jnp.float64),
    valid=jnp.asarray(valid, dtype=bool),
    pc_coords=pc_coords,
    show=True,
)


Now go back and adjust your initial guess parameters if necessary (e.g. if the parameter combination led to an error because the centrifugal radius is larger than r0).

The initial guess does not need to be good, so do not spend long on this step.

Once happy with your initial guess, continue.

## 3. Set bounds

Each optimisable parameter needs a set of bounds, with units. These are used to a) normalise the parameters so that they each contribute to a similar level in the optimisation, and b) make sure STING does not go off in completely the wrong direction.

It is recommended to define your omega bounds based on your r0 bounds to keep the centrifugal radius (r_c) shorter than r0.

In [ ]:
def get_omega(mass, r0):
    '''
    this gets value of omega when r_cent = 0.5 * r0
    '''
    omega_squared = 0.5 * G * mass / (jnp.power(r0, 3) * jnp.power(au_in_km, 2)) # in s^-2
    omega = jnp.power(omega_squared, 0.5) # in s^-1
    return omega

opt_params = initial_opt_params.copy()

# Define physically reasonable bounds (omega bounds transformed to natural log space)
# These bounds are also used as normalization anchors: x_norm = (x - min) / (max - min).
# Provide bounds for every optimized parameter.
r0_min, r0_max = 200.0, 10000.0 # param bounds in au

# the omega bounds are set by keeping centrifugal radius reasonable (r_cent = 0.5 r0)
omega_max = get_omega(fixed_params['mass'], r0_min)
omega_min = get_omega(fixed_params['mass'], r0_max)
# print these in scientific notation for sanity check
print(f"Omega bounds: {omega_min:.2e} to {omega_max:.2e} 1/s")

param_bounds = {
    'r0': (r0_min, r0_max) * u.au,                    # radius between 200-20000 au
    'theta0': (0.0, 180.0) * u.deg,                    # polar angle 0-180 degrees
    'phi0': (0.0, 360.0) * u.deg,                    # azimuthal angle 0-360 degrees
    'log_omega': (np.log(omega_min), np.log(omega_max)),  # omega in [omega_min, omega_max] 1/s
    'v_r0': (-5.0, 5.0) * u.km / u.s # km/s, 
}

## 4. Fit streamline

Choose your settings, or leave them as the defaults, and run STING.

In [ ]:

# add v_r0
log_file = 'streamfit_test_output/optimisation_log.csv'
trace_file = 'streamfit_test_output/optimisation_trace.csv'
n_epochs = 300
info_every = 10
learning_rate = 0.005 # Single learning rate applied to all normalized optimization parameters
loss_method = 1 # options: 0: LOSS_RADECVEL, 1: LOSS_RTHETAVEL

gradient_tol = 1e-2 * len(initial_opt_params) # gradient tolerance scaled by number of parameters

## here we run the fit, using cProfile to track performance
profile = False # set to True to enable cProfile profiling of the optimization run
if profile:
    profiler = cProfile.Profile()
    profiler.enable()

jax.config.update("jax_debug_nans", False)
best_opt_params, loss_history, param_errors = gradient_descent.fit_streamline(
    opt_params,
    fixed_params,
    data,
    uncertainties,
    distance,
    learning_rate=learning_rate,
    param_bounds=param_bounds,
    n_epochs=n_epochs,
    info_every=info_every,
    loss_threshold=0.05,
    loss_threshold_epochs=20,
    gradient_tol=gradient_tol,
    gradient_tol_epochs=20,
    early_stopping_patience=210,
    log_file=log_file,
    trace_file=trace_file,
    loss_method=loss_method,
    output_uncertainties=True,
 )

if profile:
    profiler.disable() # Stop profiling after optimization is complete


print(f"Optimized using loss_method='{loss_method}'")

## 5. Optional plots

### Best fit

In [ ]:
# Use only optimized parameters (exclude derived omega from best_opt_params if present)
opt_keys = list(initial_opt_params.keys())
best_for_cov = {k: float(best_opt_params[k]) for k in opt_keys}

# Prepare data-only quantities once
prepared_data = extract_streamline.prepare_data(data, uncertainties, n_elements=10)

# Recover covariance from Hessian-based uncertainty estimate
param_errors_cov, cov = errors.estimate_parameter_errors(
    best_for_cov,
    fixed_params,
    data,
    uncertainties,
    distance,
    prepared_data,
    loss_method=loss_method,
    gradient_tol=gradient_tol,
    normalisation_spec=None,
    )

param_errors_plot = {k: float(param_errors[k]) for k in opt_keys}

# ---------- 1) Normalized parameter error bars ----------
param_vals = np.array([best_for_cov[k] for k in opt_keys], dtype=float)
param_errs = np.array([param_errors_plot[k] for k in opt_keys], dtype=float)

outputs.plot_param_uncertainties(opt_keys, param_vals, param_errs, save_folder='streamfit_test_output')

# ---------- 2) Correlation heatmap from covariance ----------
# normalised correlation_{i,j} = covariance_{i,j} / (sigma_i * sigma_j)

param_names = opt_keys
outputs.plot_param_correlations(param_names, cov, save_folder='streamfit_test_output')

### Loss

In [ ]:
outputs.plot_loss(loss_history, save_folder='streamfit_test_output')

### Uncertainties

In [ ]:
# Use only optimized parameters (exclude derived omega from best_opt_params if present)
opt_keys = list(initial_opt_params.keys())
best_for_cov = {k: float(best_opt_params[k]) for k in opt_keys}

# Prepare data-only quantities once
prepared_data = extract_streamline.prepare_data(data, uncertainties, n_elements=10)

# Recover covariance from Hessian-based uncertainty estimate
param_errors_cov, cov = errors.estimate_parameter_errors(
    best_for_cov,
    fixed_params,
    data,
    uncertainties,
    distance,
    prepared_data,
    loss_method=loss_method,
    gradient_tol=gradient_tol,
    normalisation_spec=None,
    )

param_errors_plot = {k: float(param_errors[k]) for k in opt_keys}

# ---------- 1) Normalized parameter error bars ----------
param_vals = np.array([best_for_cov[k] for k in opt_keys], dtype=float)
param_errs = np.array([param_errors_plot[k] for k in opt_keys], dtype=float)

outputs.plot_param_uncertainties(opt_keys, param_vals, param_errs, save_folder='streamfit_test_output')

# ---------- 2) Correlation heatmap from covariance ----------
# normalised correlation_{i,j} = covariance_{i,j} / (sigma_i * sigma_j)

param_names = opt_keys
outputs.plot_param_correlations(param_names, cov, save_folder='streamfit_test_output')

# ---------- 3) Spaghetti
samples = outputs.sample_parameter_sets_from_covariance(
    best_for_cov,
    cov,
    opt_keys,
    param_bounds=param_bounds,
    n_samples=100,
)

streamlines = outputs.evaluate_streamlines_samples(
    samples,
    opt_keys,
    fixed_params,
    distance,
)

outputs.plot_streamline_covariance_samples(
    streamlines,
    best_for_cov,
    fixed_params,
    distance,
    data,
    uncertainties,
    velocity_reference=v_lsr,
)


### Logs

Use these to check that the optimisation has gone smoothly

In [ ]:
# import data
optimisation_log, trace_log = outputs.load_optimisation_logs('streamfit_test_output')

trace_loss_method, trace_component_columns = outputs.detect_trace_loss_method(trace_log)

# find spikes in loss over epochs
epochs = optimisation_log['epoch'].values
loss = optimisation_log['loss'].values
spikes = outputs.find_spikes(loss, threshold=0.1)
spike_epochs = epochs[spikes]

# plot parameter optimisation over epochs
outputs.plot_param_optimisation_history(optimisation_log, trace_log, trace_component_columns, spikes, save_folder='streamfit_test_output')

# plot trace diagnostics (for bug fixing)
outputs.plot_trace_diagnostics(optimisation_log, trace_log, spikes, save_folder='streamfit_test_output')

### Plots at every epoch

RA - Dec, RA - Velocity, Dec - Velocity, and Velocity - Projected Radius

Set make_video=True to also string each set of these plots into a video

In [ ]:
outputs.plot_morphology_by_epoch(
    optimisation_log=optimisation_log,
    param_names=param_names,
    gradient_descent=gradient_descent,
    fixed_params=fixed_params,
    distance=distance,
    ra_data=jnp.asarray(ra_data),
    dec_data=jnp.asarray(dec_data),
    ra_sigma=jnp.asarray(ra_sigma),
    dec_sigma=jnp.asarray(dec_sigma),
    pc_coords=jnp.asarray(pc_coords),
    n_points=n_points,
    output_dir="streamfit_test_output/epochs/morphology",
    make_video=True
)

In [ ]:
outputs.plot_ra_vel_by_epoch(
    optimisation_log=optimisation_log,
    param_names=param_names,
    gradient_descent=gradient_descent,
    fixed_params=fixed_params,
    distance=distance,
    ra_data=ra_data,
    dec_data=dec_data,
    v_data=v_data,
    ra_sigma=ra_sigma,
    v_sigma=v_sigma,
    pc_coords=pc_coords,
    output_dir="streamfit_test_output/epochs/ra_vel",
    make_video=True
)

In [ ]:
outputs.plot_dec_vel_by_epoch(
    optimisation_log=optimisation_log,
    param_names=param_names,
    gradient_descent=gradient_descent,
    fixed_params=fixed_params,
    distance=distance,
    ra_data=ra_data,
    dec_data=dec_data,
    v_data=v_data,
    dec_sigma=dec_sigma,
    v_sigma=v_sigma,
    pc_coords=pc_coords,
    output_dir="streamfit_test_output/epochs/dec_vel",
    make_video=True
)

In [ ]:
# now doing plots of velocity vs projected radius by epoch, with a KDE background

kde_background = outputs.build_velocity_radius_kde(
    ra_data,
    dec_data,
    v_data,
    grid_size=100
)

outputs.plot_vel_radius_by_epoch(
    optimisation_log=optimisation_log,
    param_names=param_names,
    gradient_descent=gradient_descent,
    fixed_params=fixed_params,
    distance=distance,
    ra_data=ra_data,
    dec_data=dec_data,
    v_data=v_data,
    ra_sigma=ra_sigma,
    dec_sigma=dec_sigma,
    v_sigma=v_sigma,
    kde_background=kde_background,
    velocity_reference=v_lsr,
    output_dir='streamfit_test_output/epochs/vel_radius',
    make_video=True
)